# Error Remediation Learning

**Last updated:** 2026-04-15 12:00 PT

**Track:** Learning | **Sub-Ability:** Associative learning (error-to-fix mapping)

Can the model learn the remediation procedures of an **invented CLI tool** from a
few worked examples, instead of hallucinating plausible-looking flags?

We introduce `zkl`, a fictional package/deploy CLI whose errors follow a
structured grammar (error-code prefix = category; keywords = specific fix).
The model studies a short troubleshooting manual (5–10 pairs), then must
emit the correct remediation command for a NEW error from the same tool.

**Failure mode probed:** hallucinated flags (e.g. `--libgs`, `--no-turbopack`)
and guessed remediations. Models that ignore the study notes and guess from
their training prior will fail; models that actually learn the mapping will
succeed.

**Difficulty grid:** complexity (direct_lookup / pattern_match / multi_factor)
× evidence (5 / 7 / 10 pairs) × 3 seeds = 27 items.

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import json
import time

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

def extract_short_answer(response: str) -> str:
    """Extract the remediation command (last short line, stripped of formatting).

    Remediations always start with 'zkl' so we prefer the last line that
    starts with 'zkl' after stripping markdown/code fences.
    """
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines:
        return ''
    # Prefer a line that starts with 'zkl'
    for line in reversed(lines):
        clean = line.strip('"\'‘’“”').rstrip('.!?')
        if clean.lower().startswith('zkl '):
            return clean.lower().strip()
    # Fallback: last short line
    for line in reversed(lines):
        clean = line.strip('"\'‘’“”').rstrip('.!?')
        if len(clean) <= 80:
            return clean.lower().strip()
    return lines[-1].strip('"\'‘’“”').rstrip('.!?').lower().strip()

In [ ]:
# === Load dataset from Kaggle ===
import os, glob
candidates = glob.glob('/kaggle/input/**/error_remediation_dataset.csv', recursive=True)
if candidates:
    csv_path = candidates[0]
else:
    csv_path = '/kaggle/input/error_remediation_dataset.csv'
print(f'Loading from: {csv_path}')
dataset = pd.read_csv(csv_path)
print(f'Dataset: {len(dataset)} items')

In [ ]:
PRE_P = """You operate an invented CLI tool called `zkl`. You are given an error message from it.

Error:
{test_input}

What is the exact remediation command?

Reply with ONLY the remediation command (a single line starting with `zkl ...`). No explanation."""

STUDY_P = """You are reading the troubleshooting manual for an invented CLI tool called `zkl`.
Each entry shows an error message and the exact remediation command that resolves it.

--- MANUAL EXCERPT ---
{material}
--- END EXCERPT ---

Create a systematic study guide for this manual:
1. Group errors by their error-code prefix family (e.g. E1xxx, E2xxx, E3xxx) and describe what each family means.
2. Identify patterns that map error features (code prefix, keywords, context qualifiers) to remediation verbs, targets, and flags.
3. For each studied pair, note WHY that remediation applies (which features of the error select it).
4. If the remediation depends on multiple factors (code family AND a context keyword), describe the joint rule explicitly.
5. Produce a concise lookup table: (error feature) -> (remediation command).

Show all work. Do not invent flags that were not present in the manual."""

POST_P = """You previously studied the `zkl` troubleshooting manual and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Original manual excerpt (for reference):
{material}

A new error has occurred:

Error:
{test_input}

What is the exact remediation command?

Reply with ONLY the remediation command (a single line starting with `zkl ...`). No explanation. Do not invent flags that do not appear in the manual."""

## Evaluation

In [ ]:
all_results = []

@kbench.task(name='error_remediation',
             description='Pre/post learning of invented-CLI error->remediation mapping')
def error_remediation(llm) -> float:
    model_name = str(llm)
    correct = 0
    total = len(dataset)

    for _, row in dataset.iterrows():
        material = row['material']
        test_input = row['test_input']
        expected = row['expected']
        complexity = row['complexity']
        evidence = row['evidence']
        difficulty_label = row['difficulty_label']
        seed = int(row['seed'])
        item_desc = row['item_desc']
        ts = time.strftime('%Y-%m-%d %H:%M:%S')
        tid = f'{complexity}_{evidence}_{seed}'

        # --- PRE: baseline without study ---
        t0 = time.time()
        pre_raw = llm.prompt(PRE_P.format(test_input=test_input))
        pre_time = time.time() - t0
        pre_extracted = extract_short_answer(pre_raw)
        pre_correct = pre_extracted == expected.lower().strip()

        # --- STUDY: create remediation study guide ---
        t0 = time.time()
        study_raw = llm.prompt(STUDY_P.format(material=material))
        study_time = time.time() - t0
        notes = strip_thinking(study_raw)

        # --- POST: recall with study notes ---
        t0 = time.time()
        post_raw = llm.prompt(POST_P.format(notes=notes, material=material, test_input=test_input))
        post_time = time.time() - t0
        post_extracted = extract_short_answer(post_raw)
        post_correct = post_extracted == expected.lower().strip()

        if post_correct:
            correct += 1

        all_results.append({
            'timestamp': ts, 'model': model_name, 'task_id': tid,
            'complexity': complexity, 'evidence': evidence, 'difficulty_label': difficulty_label,
            'seed': int(seed), 'item_desc': item_desc,
            'test_input': test_input, 'expected': expected,
            'pre_raw': pre_raw, 'pre_extracted': pre_extracted, 'pre_correct': int(pre_correct),
            'study_notes': notes,
            'post_raw': post_raw, 'post_extracted': post_extracted, 'post_correct': int(post_correct),
            'pre_time_s': pre_time, 'study_time_s': study_time, 'post_time_s': post_time
        })

        b, l = 'Y' if pre_correct else 'N', 'Y' if post_correct else 'N'
        short_test = (test_input[:50] + '...') if len(test_input) > 53 else test_input
        print(f'[{model_name:40s}] [{difficulty_label:22s}] "{short_test}" -> "{expected}"  '
              f'pre="{pre_extracted}"({b})  post="{post_extracted}"({l})')
        kbench.assertions.assert_equal(post_extracted, expected.lower().strip())

    score = correct / total
    print(f'\nScore: {correct}/{total} = {score:.4f}')
    return score

try:
    runs = error_remediation.evaluate(llm=[kbench.llm],
                                      n_jobs=1, timeout=3600, max_attempts=1)
    print(f'\nDone: {len(runs.as_dataframe())} items')
except Exception as e:
    print(f'SDK post-processing error (non-fatal): {e}')

## Results

In [ ]:
results = pd.DataFrame(all_results)
print(f'Total runs: {len(results)}\n')

pre_acc = results['pre_correct'].mean()
post_acc = results['post_correct'].mean()
gain = post_acc - pre_acc

print('=' * 60)
print(f'Model: {results["model"].iloc[0] if len(results) > 0 else "N/A"}')
print(f'Pre-learning accuracy:  {pre_acc:.1%}')
print(f'Post-learning accuracy: {post_acc:.1%}')
print(f'Learning gain:          {gain:+.1%}')
print()

print('By Difficulty:')
print('-' * 60)
for label in sorted(results['difficulty_label'].unique()):
    sub = results[results['difficulty_label'] == label]
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {label:25s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Item: {row["item_desc"]}')
    print(f'Expected: "{row["expected"]}"')
    print(f'Pre: "{row["pre_extracted"]}" ({b})  Post: "{row["post_extracted"]}" ({l})')
    print(f'Notes (first 300 chars): {row["study_notes"][:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Error Remediation: Pre vs Post-Learning')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('error_remediation_results.png', dpi=150, bbox_inches='tight')
plt.show()